## Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Libraries

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold

## Set Paths

In [3]:
# Load the dataset annotations csv file
csv_file = '/content/drive/MyDrive/Wound_Assessment/Data_Cleaning/Extracted_Images_Annotations.xlsx'
df = pd.read_excel(csv_file)

# Keep relevant columns
df = df[['filename', 'Final_Inf', 'Final_Moist', 'Final_Edge', 'original_folder']]
df['Wound_ID'] = df['original_folder'] # Map Wound_ID (for grouping in stratified 5-fold cross validation splitting)

# Load image folder
image_folder = '/content/drive/MyDrive/Wound_Assessment/Extracted_Images'

# The full image path
df['full_image_path'] = df['filename'].apply(lambda x: os.path.join(image_folder, x))

## Data Checks

In [4]:
# Check for duplicate filenames in the dataset
duplicate_filenames = df['filename'].duplicated().sum()
print(f"Number of duplicate filenames: {duplicate_filenames}")

# Check for missing paths
invalid_paths = df['full_image_path'].apply(lambda x: not os.path.exists(x)).sum()
print(f"Number of invalid image paths: {invalid_paths}")

Number of duplicate filenames: 0
Number of invalid image paths: 107


## Stratified Group 5-Fold Split

The split uses `Wound_ID` as the group, so images from the same wound stay in the same fold.

In [5]:
df = df.sort_values(by='filename').reset_index(drop=True)

# Create temporary label combination keys
df['stratify_key'] = (
    df['Final_Inf'].astype(str) + "_" +
    df['Final_Moist'].astype(str) + "_" +
    df['Final_Edge'].astype(str)
)

print(f"Unique Label Combinations: {df['stratify_key'].nunique()}")

sgkf = StratifiedGroupKFold(n_splits=5)

# Prepare Inputs
X = df['filename'].values
y = df['stratify_key'].values
groups = df['Wound_ID'].values

# Initialize fold column
df['fold'] = -1

# Loop
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    print(f"\n Fold {fold + 1} ")

    # Save fold info (So we have a permanent record)
    df.loc[val_idx, 'fold'] = fold

    # Create subsets for inspection
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    # Verify Balance
    print(f"Train Size: {len(train_df)} | Val Size: {len(val_df)}")

    # Check if we have at least one sample of each Moisture class (0, 1, 2)
    inf_classes = val_df[val_df['Final_Inf'] != -1]['Final_Inf'].nunique()
    print(f"  > Infection Classes Present: {inf_classes}/2")

    moist_classes = val_df[val_df['Final_Moist'] != -1]['Final_Moist'].nunique()
    print(f"  > Moisture Classes Present: {moist_classes}/3")

    edge_classes = val_df[val_df['Final_Edge'] != -1]['Final_Edge'].nunique()
    print(f"  > Edge Classes Present: {edge_classes}/2")

    # Reset Index
    train_df.reset_index(drop=True, inplace=True)
    val_df.reset_index(drop=True, inplace=True)

    # Leakage Check
    overlap = set(train_df['Wound_ID']).intersection(set(val_df['Wound_ID']))
    print(f"  > Patient Leakage: {len(overlap)} (Must be 0)")

# Clean up and Save
# Remove the temporary key column before saving
df.drop(columns=['stratify_key'], inplace=True)

save_path = '/content/drive/MyDrive/Wound_Assessment/metadata_with_folds.csv'
df.to_csv(save_path, index=False)
print(f"\n Saved to: {save_path}")

Unique Label Combinations: 15

 Fold 1 
Train Size: 89 | Val Size: 18
  > Infection Classes Present: 2/2
  > Moisture Classes Present: 3/3
  > Edge Classes Present: 2/2
  > Patient Leakage: 0 (Must be 0)

 Fold 2 
Train Size: 89 | Val Size: 18
  > Infection Classes Present: 2/2
  > Moisture Classes Present: 3/3
  > Edge Classes Present: 2/2
  > Patient Leakage: 0 (Must be 0)

 Fold 3 
Train Size: 89 | Val Size: 18
  > Infection Classes Present: 2/2
  > Moisture Classes Present: 3/3
  > Edge Classes Present: 2/2
  > Patient Leakage: 0 (Must be 0)

 Fold 4 
Train Size: 72 | Val Size: 35
  > Infection Classes Present: 2/2
  > Moisture Classes Present: 3/3
  > Edge Classes Present: 2/2
  > Patient Leakage: 0 (Must be 0)

 Fold 5 
Train Size: 89 | Val Size: 18
  > Infection Classes Present: 2/2
  > Moisture Classes Present: 3/3
  > Edge Classes Present: 2/2
  > Patient Leakage: 0 (Must be 0)

 Saved to: /content/drive/MyDrive/Wound_Assessment/metadata_with_folds_handover.csv


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
